In [ ]:
# --- ADDING SERIES FEATURE ---
# As mentioned in the report, series books have different endurance patterns.
# We identify them by looking for the '#' symbol in the title (e.g., "The Hunger Games #1")
df['is_series'] = df['title'].str.contains('#').astype(int)

print(f"Series books identified: {df['is_series'].sum()}")

: 

In [ ]:
# 1. Load the master dataset from Notebook 3
df = pd.read_csv('data/final_modeling_data.csv')

# --- ADDING SERIES FEATURE ---
# As discussed in the report, series books have different endurance patterns.
# We identify them by looking for the '#' symbol in the title.
df['is_series'] = df['title'].str.contains('#').astype(int)

# 2. Define the exact feature list from the report methodology
features = [
    'author_reputation', 
    'is_series',          
    'avg_sentiment', 
    'review_count', 
    'ratings_count',
    'average_rating',
    'publication_year'
]

# 3. Create X (features) and y (target)
X = df[features]
y = df['bestseller']

print(f"Features being used: {X.columns.tolist()}")
print(f"Total Series books identified: {df['is_series'].sum()}")

In [ ]:
# Imports and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
import lightgbm as lgb

# Load the master dataset from Notebook 3
df = pd.read_csv('data/final_modeling_data.csv')

# 1. Load the master dataset from Notebook 3
df = pd.read_csv('data/final_modeling_data.csv')

# 2. Identify columns that actually exist in the dataframe
# We define what we WANT to drop, but only if they are present
potential_drops = ['book_id', 'title', 'authors', 'author_id', 'publisher', 'title_clean', 'description']
existing_drops = [c for c in potential_drops if c in df.columns]

# 3. Create X (features) and y (target)
X = df.drop(columns=existing_drops + ['bestseller'])
y = df['bestseller']

# --- VALIDATION ---
print(f"Features being used: {X.columns.tolist()}")
print(f"Bestseller target confirmed: {y.name}")

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data split successfully!")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# The "Safety Imputer"
# Check which columns have NaNs (just for your own info)
print("Columns with missing values:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Fill all remaining NaNs in X_train and X_test with 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("\nNaNs cleared. Re-running Logistic Regression...")

Every good project needs a baseline to prove the advanced model is actually better.

In [ ]:
# Baseline Model (Logistic Regression)
print("Training Logistic Regression Baseline...")
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)
baseline_probs = baseline.predict_proba(X_test)[:, 1]

print(f"Baseline AUC-ROC: {roc_auc_score(y_test, baseline_probs):.4f}")
print(f"Baseline F1-Score: {f1_score(y_test, baseline_preds):.4f}")

This is the core model from our group project. We use scale_pos_weight to handle the class imbalance (since most books are not bestsellers).

In [ ]:
# The Primary Model (LightGBM)
# Calculate the ratio for imbalance handling
ratio = (len(y_train) - y_train.sum()) / y_train.sum()

lgbm_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=ratio, # Critical for imbalanced data
    random_state=42
)

lgbm_model.fit(X_train, y_train)
lgbm_probs = lgbm_model.predict_proba(X_test)[:, 1]

print(f"LightGBM AUC-ROC: {roc_auc_score(y_test, lgbm_probs):.4f}")

Our report highlights Precision@100. This is a business-centric metric: "If we recommend the top 100 books the model is most confident in, how many are actually bestsellers?"

In [ ]:
# Precision@K Evaluation
def precision_at_k(y_true, y_probs, k):
    # Sort by probability in descending order
    df_results = pd.DataFrame({'true': y_true, 'prob': y_probs})
    top_k = df_results.sort_values(by='prob', ascending=False).head(k)
    return top_k['true'].mean()

p100 = precision_at_k(y_test, lgbm_probs, 100)
p500 = precision_at_k(y_test, lgbm_probs, 500)

print(f"Precision@100: {p100:.2%}")
print(f"Precision@500: {p500:.2%}")

In [ ]:
# Feature Importance Visualization
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgbm_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feat_imp.head(15))
plt.title('Top 15 Predictors of NYT Bestseller Status')
plt.show()

In [ ]:
def calculate_precision_at_k(y_true, y_probs, k=100):
    # Combine actuals and probabilities into a DataFrame
    results = pd.DataFrame({'actual': y_true, 'prob': y_probs})
    # Sort by probability (highest first) and take the top K
    top_k = results.sort_values(by='prob', ascending=False).head(k)
    # Precision is (correct predictions in top K) / K
    precision = top_k['actual'].sum() / k
    return precision

# Get probabilities from your trained LightGBM model
y_probs = lgb_model.predict(X_test)

# Calculate and print
p100 = calculate_precision_at_k(y_test, y_probs, k=100)
print(f"--- REPORT ALIGNMENT CHECK ---")
print(f"Precision@100: {p100:.4f}")
print(f"Target from Report: 0.9600")